In [4]:
!pip install bs4


   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [bs4]




[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests
import sqlite3
import pandas as pd
from bs4 import BeautifulSoup

In [6]:
url = 'https://web.archive.org/web/20230908091635%20/https://en.wikipedia.org/wiki/List_of_largest_banks'
db_name = 'Banks.db'
table_name = 'Largest_banks'
table_attribs = ['Name', 'MC_USD_Billion']
csv_path = './Largest_banks_data.csv'

# ETL FUNCTION LOG

In [7]:
from datetime import datetime 

def log_progress(message):
    timestamp_format = '%Y-%h-%d-%H:%S'
    now = datetime.now()
    timestamp = now.strftime(timestamp_format)
    with open("./etl_project_log.txt", "a") as f:
        f.write(timestamp + ' : ' + message + '\n')

# EXTRACTION SESSION

In [8]:
def extract(url, table_attribs):
    page = requests.get(url).text
    data = BeautifulSoup(page, 'html.parser')
    df = pd.DataFrame(columns=table_attribs)
    tables = data.find_all('tbody')
    rows = tables[2].find_all('tr')

    for row in rows:
        col = row.find_all('td')
        if len(col) !=0:
            if col[0].find('a') is not None and '—' not in col[1]:
                data_dict = {"Name": col[0].a.contents[0],
                             "MC_USD_Billion": col[1].contents[0]}
                df1 =  pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df, df1], ignore_index=True)
    return df

# Transform Information

In [9]:
def transform(df):
    exchange_rate_df = pd.read_csv('exchange_rate.csv')
    exchange_dict = exchange_rate_df.set_index('Currency').to_dict()['Rate']
    df['MC_USD_Billion'] = pd.to_numeric(df['MC_USD_Billion'].astype(str).str.replace(',', ''), errors='coerce')

    df['MC_GBP_Billion'] = [round(x * exchange_dict['GBP'], 2) for x in df['MC_USD_Billion']]
    df['MC_EUR_Billion'] = [round(x * exchange_dict['EUR'], 2) for x in df['MC_USD_Billion']]
    df['MC_INR_Billion'] = [round(x * exchange_dict['INR'], 2) for x in df['MC_USD_Billion']]
    
    return df
    # df.to_csv('transformed_data.csv', index=False)

# Data Loading

In [10]:
def load_to_csv(df, csv_path):
    df.to_csv(csv_path)

def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

# Querying Databases

In [11]:
def run_query(query_statement, sql_connection):
    print(query_statement)
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)

# FUNCTION CALLS

In [12]:
log_progress('Preliminaries complete. Initiating ETL process')

df = extract(url, table_attribs)

log_progress('Data extraction complete. Initiating Transformation process')

df = transform(df)

log_progress('Data transformation complete. Initiating loading process')

load_to_csv(df, csv_path)

log_progress('Data saved to CSV file')

sql_connection = sqlite3.connect('Banks.db')

log_progress('SQL Connection initiated.')

load_to_db(df, sql_connection, table_name)

log_progress('Data loaded to Database as table. Running the query')

query_statement = f"SELECT * from {table_name} WHERE MC_USD_Billion >= 100"
run_query(query_statement, sql_connection)

log_progress('Process Complete.')

sql_connection.close()

SELECT * from Largest_banks WHERE MC_USD_Billion >= 100
Empty DataFrame
Columns: [Name, MC_USD_Billion, MC_GBP_Billion, MC_EUR_Billion, MC_INR_Billion]
Index: []


In [13]:
sql_connection = sqlite3.connect('Banks.db')

In [17]:
query_output_1 = pd.read_sql('SELECT * FROM Largest_banks', sql_connection)
print(query_output_1)
query_output_2 = pd.read_sql('SELECT AVG(MC_GBP_Billion) FROM Largest_banks', sql_connection)
print(query_output_2)
query_output_3 = pd.read_sql('select Name, MC_USD_Billion, MC_GBP_Billion, MC_EUR_Billion,  MC_INR_Billion  from Largest_banks order by MC_USD_Billion desc  LIMIT 6', sql_connection)
print(query_output_3)

                    Name  MC_USD_Billion  MC_GBP_Billion  MC_EUR_Billion  \
0                 France               6             4.8            5.58   
1                 Canada               6             4.8            5.58   
2            South Korea               6             4.8            5.58   
3            Netherlands               3             2.4            2.79   
4              Singapore               3             2.4            2.79   
5                 Brazil               3             2.4            2.79   
6                  Italy               2             1.6            1.86   
7                 Russia               2             1.6            1.86   
8                 Sweden               2             1.6            1.86   
9                Finland               1             0.8            0.93   
10               Denmark               1             0.8            0.93   
11               Belgium               1             0.8            0.93   
12          